In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.core.interchange.dataframe_protocol import DataFrame

np.random.seed(42)

TASK1

In [2]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    data = np.asarray(data)
    boot_stats = np.zeros(n_boot)
    for i in range(n_boot):
        boot_sample = np.random.choice(data, size=len(data), replace=True)
        boot_stats[i] = stat_func(boot_sample)

    alpha = 1 - ci_level
    ci_lower = np.percentile(boot_stats, 100 * alpha / 2)
    ci_upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))

    return ci_lower, ci_upper


In [3]:
data=np.arange(1,101)
print(bootstrap_ci(data))

(np.float64(44.799749999999996), np.float64(56.100249999999996))


TASK2

In [4]:
income=np.random.normal(loc=50, scale=10, size=250)
purchased=np.random.binomial(n=1, p=0.5, size=250)
a={"Income":income,
   "Purchased":purchased}

df=pd.DataFrame(a)

In [5]:
df

,Income,Purchased
0,35.985999,0
1,50.452223,1
2,52.043324,1
3,46.679720,1
4,47.135542,1
...,...,...
245,46.840167,0
246,53.872696,0
247,54.979662,0
248,66.109671,0


In [6]:
overall = pd.DataFrame()
overall["Mean (Income)"] = bootstrap_ci(df["Income"], np.mean)
overall["Median (Income)"] = bootstrap_ci(df["Income"], np.median)
overall["Proportion (Binary)"] = bootstrap_ci(df["Purchased"], np.mean)

summary_table = overall.T
summary_table.columns = ["Lower CI", "Upper CI"]
display(summary_table)

,Lower CI,Upper CI
Mean (Income),49.233104,51.732764
Median (Income),49.234353,52.026850
Proportion (Binary),0.452000,0.576000


In [7]:
overall

,Mean (Income),Median (Income),Proportion (Binary)
0,49.233104,49.234353,0.452
1,51.732764,52.026850,0.576


TASK3

In [8]:
#Normal-Approximation for Mean (using t-distribution)
mean_val = df["Income"].mean()
sem_val = st.sem(df["Income"])
norm_mean_ci = st.t.interval(0.95, len(df["Income"])-1, loc=mean_val, scale=sem_val)

#Normal-Approximation for Proportion (Wald Interval)
p_hat = df["Purchased"].mean()
n = len(df["Purchased"])
z = 1.96 
margin_error = z * np.sqrt((p_hat * (1 - p_hat)) / n)
norm_prop_ci = (p_hat - margin_error, p_hat + margin_error)

#Comparison Table 
comparison_df = pd.DataFrame({
    "Statistic": ["Mean", "Proportion"],
    "Bootstrap Lower": [overall["Mean (Income)"].iloc[0], overall["Proportion (Binary)"].iloc[0]],
    "Bootstrap Upper": [overall["Mean (Income)"].iloc[1], overall["Proportion (Binary)"].iloc[1]],
    "Normal Lower": [norm_mean_ci[0], norm_prop_ci[0]],
    "Normal Upper": [norm_mean_ci[1], norm_prop_ci[1]]
})

display(comparison_df)

,Statistic,Bootstrap Lower,Bootstrap Upper,Normal Lower,Normal Upper
0,Mean,49.233104,51.732764,49.226293,51.760413
1,Proportion,0.452000,0.576000,0.454051,0.577949


Yes, the two approaches are giving extremely similar intervals.For the Mean, the Bootstrap range is $[49.23, 51.73]$ compared to the Normal range of $[49.22, 51.76]$.For the Proportion, the Bootstrap range is $[0.45, 0.57]$ compared to the Normal range of $[0.45, 0.57]$.They would typically diverge if the sample size was much smaller (like $n < 30$) or if the underlying data was heavily skewed with extreme outliers. In those cases, the Normal-Approximation (which assumes a symmetric, bell-shaped sampling distribution) would be less accurate than the Bootstrap, which adapts to the data's actual shape.

The bootstrap approach is especially useful for the Median.While the Mean and Proportion have standard formulas for calculating Standard Error (like the $t$-distribution or Wald formula), the Median does not have a straightforward, universal formula for its confidence interval. The Bootstrap is "distribution-free," meaning it can calculate a CI for the median—or any other complex statistic—just as easily as it does for the mean, without requiring us to assume the data follows a specific mathematical distribution.